In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device()#['cuda:1', 'cuda:0'])
# device, generator = u.Devices().set_device('cpu')

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=False, 
    log_transform=False,
    scale_method=None,

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


In [2]:
#### convenience variables ####
_embedding_size = 16

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch (forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features
_x = _batch.x.view(_batch.batch_size, int(_batch.x.shape[0]/_batch.batch_size), -1)

In [3]:
_batch.to_dict().keys()

dict_keys(['x', 'edge_index', 'edge_attr', 'y', 'sample_id', 'laplacian_pe', 'batch', 'ptr'])

In [4]:
_batch.edge_attr

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.]], device='cuda:4')

# graph
---

In [36]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch import nn

from torch import Tensor
from typing import Literal, Optional

from modules.model import SequentialModel


In [56]:
class TanhGATConv(MessagePassing):
    def __init__(self, in_channels:int, out_channels:int, edge_channels:Optional[int]=None, temperature:float=1.0, eps=1e-8, *args, **kwargs):
        super().__init__(aggr='add')
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.temperature = temperature
        self.eps = eps

        self.lin_q = nn.Linear(in_channels, out_channels)
        self.lin_kv = nn.Linear(in_channels, 2*out_channels)
        self.norm = nn.LayerNorm(out_channels)

        if edge_channels is not None:
            self.lin_e = nn.Linear(edge_channels, 6*out_channels)
        else:
            self.lin_e = None

    def forward(self, x:Tensor, edge_index:Tensor, edge_attr:Optional[Tensor]=None, *args, **kwargs):
        # get qkv
        q = self.lin_q(x)
        k, v = self.lin_kv(x).chunk(2, dim=-1)

        # transformer-like attention module
        out = self.propagate(edge_index, q=q, k=k, v=v, edge_attr=edge_attr)

        # add & norm
        out = self.norm(out + q)

        return out

    def message(self, q_i, k_j, v_j, edge_attr, index, size_i,):
        # get e_qkv (if applicable)
        if edge_attr is not None:
            # compute modulation values from edges
            q_e, k_e, v_e, q_gate, k_gate, v_gate = self.lin_e(edge_attr).chunk(6, dim=-1)

            # edge gated node modulation
            q_i = q_i + q_e * torch.sigmoid(q_gate)
            k_j = k_j + k_e * torch.sigmoid(k_gate)
            v_j = v_j + v_e * torch.sigmoid(v_gate)

        # vaswani dot product attention
        att = (q_i * k_j).sum(dim=-1) / self.out_channels

        # tanh L1 norm attention
        att = torch.tanh(att/self.temperature)
        att_sum = torch.zeros(size_i, device=att.device).scatter_add(0, index, att.abs())
        att = att / (att_sum[index] + self.eps)

        # save attention for analyses
        self._prev_att = att.detach()

        # apply attention to message
        return att.unsqueeze(-1) * v_j


In [ ]:
# gat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=_embedding_size,
#     edge_channels=brca.num_edge_features
# )

# _node_emb = gat(_batch.x, _batch.edge_index, _batch.edge_attr)
# print(_node_emb.shape)

# set pooling
---

In [ ]:
def handle_transformer_dims(embed_dim:Optional[int], head_dim:Optional[int], num_heads:int):
    # none specified; assert error
    assert (embed_dim is not None) or (head_dim is not None), 'one of [embed_dim, head_dim] must be specified'

    # both specified; lin_out reshapes head to embed
    if (embed_dim is not None) and (head_dim is not None):
        assert embed_dim // num_heads == head_dim, 'transformer dims incompatible, (embed_dim // num_heads == head_dim) must be true'
        return embed_dim, head_dim

    # embed_dim specified; head = embed / num_heads
    elif embed_dim is not None:
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        head_dim = embed_dim // num_heads

    # head_dim specified; embed = head * num_heads
    elif head_dim is not None:
        embed_dim = head_dim * num_heads

    return embed_dim, head_dim

In [ ]:
class AttnSetPooling(nn.Module):
    def __init__(self, mask:Tensor, pool_method:Literal['add','mean']='mean', embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, *args, **kwargs):
        '''
        mask in (nodes, sets)
        '''
        super().__init__(*args, **kwargs)
        self.mask = mask.T # convert to (sets, nodes) for attn_mask in (q, kv)
        self.num_sets, self.num_nodes = self.mask.shape
        self.pool_method = pool_method

        # sum across nodes, in (sets, 1)
        self.set_counts = self.mask.sum(dim=-1, keepdim=True).clamp(min=1) # clamp to prevent div0
        
        embed_dim, _ = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x:Tensor):
        '''
        x in (b*n,f) or (b,n,F)
        attn_mask in (n, s) or (b * heads, n, s)
        '''
        # convert x to (b,n,F) if applicable
        if x.dim() == 2:
            batch_size = int(x.shape[0]//self.num_nodes)
            x = x.view(batch_size, self.num_nodes, -1)

        # pool q
        q = self.mask @ x # (b,s,n) @ (b,n,E) -> (b,s,E)
        if self.pool_method == 'mean':
            q = q / self.set_counts

        # get attn_mask
        attn_mask = (self.mask == 0)

        # get output
        attn_output, _ = self.attn(
            query = q,
            key = x,
            value = x,
            attn_mask = attn_mask,
            need_weights = False
        )

        # add & norm
        attn_output = self.norm(attn_output + q)

        return attn_output # (b,s,E)
        


In [ ]:
# gat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=_embedding_size,
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size
# )

# _node_emb = gat(_batch.x, _batch.edge_index, _batch.edge_attr)

# _set_emb = lasp(_node_emb)
# print(_set_emb.shape)

# global pooling
---

In [34]:
class AttnGlobalPooling(nn.Module):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)

        embed_dim, _ = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

    def forward(self, x:Tensor):
        '''
        x in (b,n,E) or (b,s,E)
        '''
        batch_size = x.shape[0]

        # expand cls
        cls = self.cls_token.expand(batch_size, -1, -1)

        # get cross attn, in (b,1,E)
        attn_output, _ = self.attn(
            query = cls,
            key = x,
            value = x,
            need_weights = False
        )

        # squeeze (b,1,E) -> (b,E)
        attn_output = attn_output.squeeze(1)

        return attn_output

        

In [ ]:
# tgat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=_embedding_size,
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size
# )

# agp = AttnGlobalPooling(
#     head_dim=_embedding_size
# )

# _node_emb = tgat(_batch.x, _batch.edge_index, _batch.edge_attr)
# _set_emb = asp(_node_emb)
# _samp_emb = agp(_set_emb)
# print(_samp_emb.shape)

torch.Size([64, 16])


# lazy zinb decoder
---

In [92]:
class LazyZINBDecoder(nn.Module):
    def __init__(
        self, 
        mask:Tensor,

        embed_dim:int=None,
        head_dim:int=None, 
        num_heads:int=1,

        expand_layers:list[int]=[],
        zinb_layers:list[int]=[],

        lin_kwargs:dict={},
        zinb_kwargs:dict={},

        eps:float=1e-6,
        *args, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.eps = eps
        self.mask = mask.T.unsqueeze(0) # (1, sets, nodes)
        _, self.num_sets, self.num_nodes = self.mask.shape
        embed_dim, _ = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.expand_sample = SequentialModel(
            in_channels = embed_dim,
            out_channels = embed_dim * self.num_nodes,
            layer_class=nn.Linear,
            hidden_dims = expand_layers,
            **lin_kwargs
        )

        self.zinb = SequentialModel(
            in_channels = embed_dim,
            out_channels = 3,
            layer_class = nn.Linear,
            hidden_dims = zinb_layers,
            **zinb_kwargs
        )

    def forward(self, z:Tensor, return_dict:bool=True, *args, **kwargs):
        '''
        samp_emb (z) in (b,E)
        '''
        batch_size = z.shape[0]

        # transform sample
        emb = self.expand_sample(z).view(batch_size, self.num_nodes, -1)

        # estimate params
        mu, theta, pi = self.zinb(emb).chunk(3, dim=-1)

        # get params
        mu = F.softplus(mu) + self.eps
        theta = F.softplus(theta) + self.eps
        pi = torch.sigmoid(pi)

        # clamp for stability
        mu = torch.clamp(mu, min=self.eps)
        theta = torch.clamp(theta, min=self.eps)
        pi = torch.clamp(pi, min=self.eps, max=1.0-self.eps)

        # reconstruct x
        x_recon = (1-pi)*mu

        return x_recon, mu, theta, pi

# autoencoder
---

In [93]:
class SimpleAutoencoder(nn.Module):
    def __init__(
        self,
        # required
        in_features:int,
        out_features:int,
        mask:Tensor, # (n,s)
        decoder_class,

        # transformer
        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:float=0.0,

        # other
        edge_features:Optional[int]=None,
        temperature:int=1.0,
        pool_method:Literal['add','mean'] = 'mean',
        decoder_dims:list[int, None]=[],
        decoder_kwargs:dict={},
        *args, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_nodes = mask.shape[0]
        embed_dim, head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.node_encoder = TanhGATConv(
            in_channels=in_features,
            out_channels=embed_dim,
            edge_channels=edge_features,
            temperature=temperature
        )

        self.set_pooling = AttnSetPooling(
            mask=mask,
            pool_method=pool_method,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.global_pooling = AttnGlobalPooling(
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.decoder = LazyZINBDecoder(
            mask=mask,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            **decoder_kwargs
        )

    def forward(self, x:Tensor, *args, **kwargs):
        batch_size = int(x.shape[0]/self.num_nodes)

        node_emb = self.node_encoder(x, *args, **kwargs)
        set_emb = self.set_pooling(node_emb)
        z = self.global_pooling(set_emb)
        x_recon = self.decoder(z)

        return x_recon#.view(batch_size * self.num_nodes, 1)

In [94]:
simple_ae = SimpleAutoencoder(
    in_features=brca.num_node_features,
    out_features=brca.num_nodes,
    mask=brca.mask,
    decoder_class=nn.Linear,
    head_dim=8,
    num_heads=2
)

_x_recon, _mu, _theta, _pi = simple_ae(_batch.x, edge_index=_batch.edge_index)
print(
    _x_recon.shape,
    _mu.shape,
    _theta.shape,
    _pi.shape
)

torch.Size([64, 4383, 1]) torch.Size([64, 4383, 1]) torch.Size([64, 4383, 1]) torch.Size([64, 4383, 1])


In [96]:
_batch.x.shape

torch.Size([280512, 1])

---